# BIST AI Radar V8.1 Colab

Bu notebook BIST AI Radar motorunu Colab üzerinde çalıştırmak için hazırlanmıştır.

> Not: Varsayılan akış demo finansal tablolarla çalışır. Gerçek BIST veri sağlayıcı entegrasyonu için `BistProvider.fetch()` fonksiyonu bağlanmalıdır.


## 1. Kurulum

In [ ]:
!git clone https://github.com/ErgunUral/bist-ai-radar.git
%cd bist-ai-radar
!pip install -r requirements.txt


## 2. Modülleri Yükle

In [ ]:
from src.analyzer import analyze_statements
from src.formatter import format_analysis_result
from src.sample_data import build_sample_statements
from src.sheets_dashboard import prepare_main_dashboard, top_n, risky_stocks

print('BIST AI Radar V8.1 hazır')


## 3. Tek Hisse Demo Test

In [ ]:
ticker = 'ASELS'
result = analyze_statements(ticker, build_sample_statements(), market_value=1000)
print(format_analysis_result(result))


## 4. Toplu Hisse Demo Analizi

In [ ]:
import pandas as pd

tickers = ['ASELS', 'THYAO', 'SISE', 'KCHOL', 'AKBNK']
rows = []
for ticker in tickers:
    result = analyze_statements(ticker, build_sample_statements(), market_value=1000)
    rows.append({
        'Hisse': result.ticker,
        'Dönem': result.period,
        'Sektör Tipi': result.sector_type,
        'Genel Skor': result.total_score,
        'AI Karar': result.decision,
        'Risk': result.risk,
        'AI Uyarı': result.warnings,
        'ROE %': result.metrics.get('roe'),
        'ROA %': result.metrics.get('roa'),
        'Net Kar Marjı %': result.metrics.get('net_margin'),
        'Cari Oran': result.metrics.get('current_ratio'),
        'Borç/Özkaynak': result.metrics.get('debt_to_equity'),
        'F/K': result.metrics.get('price_to_earnings'),
        'PD/DD': result.metrics.get('price_to_book'),
        'FD/FAVÖK': result.metrics.get('ev_to_ebitda'),
    })

df = pd.DataFrame(rows)
dashboard = prepare_main_dashboard(df)
dashboard


## 5. En Güçlü 20 ve Riskli Hisseler

In [ ]:
top_20 = top_n(dashboard, 20)
risk = risky_stocks(dashboard, 40)
display(top_20)
display(risk)


## 6. Google Sheets'e Yazma

Aşağıdaki hücre Google hesabı ile yetkilendirme ister. `SPREADSHEET_NAME` değerini kendi e-tablonun adıyla değiştir.

In [ ]:
# Google Sheets yazımı için isteğe bağlı hücre
# from google.colab import auth
# import gspread
# from google.auth import default
# from src.sheets_dashboard import dataframe_to_values
#
# auth.authenticate_user()
# creds, _ = default()
# gc = gspread.authorize(creds)
#
# SPREADSHEET_NAME = 'BIST AI Radar Dashboard'
# sh = gc.open(SPREADSHEET_NAME)
# ws = sh.worksheet('Göstergeler')
# ws.clear()
# ws.update(dataframe_to_values(dashboard))
# print('Google Sheets güncellendi')


## 7. Gerçek Veri Sağlayıcı Bağlama Noktası

Gerçek BIST verisi için `src/bist_provider.py` içindeki `BistProvider.fetch()` fonksiyonu bilanço, gelir tablosu ve nakit akış DataFrame'leri dönecek şekilde bağlanmalıdır.

Dönen nesne formatı:

```python
FinancialStatements(
    balance_sheet=balance_sheet_df,
    income_statement=income_statement_df,
    cash_flow=cash_flow_df,
)
```
